<a href="https://colab.research.google.com/github/Elamathi995/Multiple-error-handling-agent/blob/main/Multiple_error_handling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ============================================================
# AGENTIC AI DRUG DISCOVERY PIPELINE
# Multi-Tool Error Handling Demo
# Google Colab Single Script
# ============================================================

# Install RDKit in Google Colab
!pip install rdkit -q

# ============================================================
# IMPORT LIBRARIES
# ============================================================

import random
import time
import logging
import pandas as pd

from rdkit import Chem
from rdkit.Chem import Descriptors

# ============================================================
# CONFIGURE LOGGING
# ============================================================

logging.basicConfig(
    filename='drug_screening.log',
    level=logging.ERROR,
    format='%(asctime)s %(levelname)s:%(message)s'
)

print("Logging initialized")

# ============================================================
# SAMPLE MOLECULES
# Some are intentionally invalid
# ============================================================

smiles_list = [
    "CCO",                        # Ethanol
    "CCN(CC)CC",                  # Triethylamine
    "INVALID_SMILES",             # Invalid molecule
    "CC(=O)Oc1ccccc1C(=O)O",      # Aspirin
    "C1CCCCC1",                   # Cyclohexane
    "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O"  # Ibuprofen
]

# ============================================================
# TOOL 1: MOLECULE VALIDATION
# ============================================================

def validate_molecule(smiles):

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        raise ValueError("Invalid SMILES structure")

    return mol

# ============================================================
# TOOL 2: DESCRIPTOR CALCULATION
# ============================================================

def calculate_descriptors(mol):

    mw = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    h_donors = Descriptors.NumHDonors(mol)
    h_acceptors = Descriptors.NumHAcceptors(mol)

    return {
        "MolecularWeight": round(mw, 2),
        "LogP": round(logp, 2),
        "HDonors": h_donors,
        "HAcceptors": h_acceptors
    }

# ============================================================
# TOOL 3: DOCKING SIMULATION
# Random failures included intentionally
# ============================================================

def run_docking(smiles):

    # Simulate docking engine failure
    if random.random() < 0.3:
        raise RuntimeError("Docking engine crashed")

    docking_score = round(random.uniform(-12, -5), 2)

    return docking_score

# ============================================================
# TOOL 4: TOXICITY PREDICTION
# ============================================================

def predict_toxicity(smiles):

    # Simulate ML model failure
    if random.random() < 0.2:
        raise RuntimeError("Toxicity prediction model failed")

    toxicity = random.choice([
        "Low",
        "Moderate",
        "High"
    ])

    return toxicity

# ============================================================
# TOOL 5: LIPINSKI RULE CHECK
# ============================================================

def lipinski_rule(descriptors):

    mw = descriptors["MolecularWeight"]
    logp = descriptors["LogP"]
    h_donors = descriptors["HDonors"]
    h_acceptors = descriptors["HAcceptors"]

    passed = (
        mw <= 500 and
        logp <= 5 and
        h_donors <= 5 and
        h_acceptors <= 10
    )

    return "Pass" if passed else "Fail"

# ============================================================
# AGENTIC AI SCREENING PIPELINE
# ============================================================

def screening_agent(smiles):

    results = {
        "SMILES": smiles
    }

    # --------------------------------------------------------
    # STEP 1: VALIDATION
    # --------------------------------------------------------

    try:

        mol = validate_molecule(smiles)

        results["Validation"] = "Passed"

    except Exception as e:

        logging.error(
            f"Validation failed for {smiles}: {e}"
        )

        results["Validation"] = "Failed"
        results["Error"] = str(e)

        return results

    # --------------------------------------------------------
    # STEP 2: DESCRIPTOR CALCULATION
    # --------------------------------------------------------

    try:

        descriptors = calculate_descriptors(mol)

        results["Descriptors"] = descriptors

    except Exception as e:

        logging.error(
            f"Descriptor calculation failed for {smiles}: {e}"
        )

        results["Descriptor_Error"] = str(e)

    # --------------------------------------------------------
    # STEP 3: LIPINSKI RULE CHECK
    # --------------------------------------------------------

    try:

        lipinski = lipinski_rule(descriptors)

        results["LipinskiRule"] = lipinski

    except Exception as e:

        logging.error(
            f"Lipinski rule check failed for {smiles}: {e}"
        )

        results["Lipinski_Error"] = str(e)

    # --------------------------------------------------------
    # STEP 4: DOCKING WITH RETRIES
    # --------------------------------------------------------

    max_retries = 3

    for attempt in range(max_retries):

        try:

            docking_score = run_docking(smiles)

            results["DockingScore"] = docking_score
            results["DockingStatus"] = "Success"

            break

        except Exception as e:

            logging.error(
                f"Docking failed for {smiles} "
                f"Attempt {attempt+1}: {e}"
            )

            print(
                f"Retrying docking for {smiles} "
                f"(Attempt {attempt+1})..."
            )

            time.sleep(2)

            if attempt == max_retries - 1:

                results["DockingStatus"] = "Failed"
                results["Docking_Error"] = str(e)

    # --------------------------------------------------------
    # STEP 5: TOXICITY PREDICTION
    # --------------------------------------------------------

    try:

        toxicity = predict_toxicity(smiles)

        results["Toxicity"] = toxicity

    except Exception as e:

        logging.error(
            f"Toxicity prediction failed for {smiles}: {e}"
        )

        results["Toxicity_Error"] = str(e)

    return results

# ============================================================
# BATCH SCREENING
# ============================================================

all_results = []

print("\n==============================")
print("STARTING DRUG SCREENING")
print("==============================")

for smiles in smiles_list:

    print(f"\nProcessing Molecule: {smiles}")

    output = screening_agent(smiles)

    all_results.append(output)

    print("Result:")
    print(output)

# ============================================================
# CONVERT RESULTS TO DATAFRAME
# ============================================================

df = pd.DataFrame(all_results)

print("\n==============================")
print("FINAL RESULTS TABLE")
print("==============================")

display(df)

# ============================================================
# SAVE RESULTS
# ============================================================

df.to_csv("drug_screening_results.csv", index=False)

print("\nResults saved as:")
print("drug_screening_results.csv")

# ============================================================
# DISPLAY LOG FILE CONTENT
# ============================================================

print("\n==============================")
print("ERROR LOG CONTENT")
print("==============================")

try:

    with open("drug_screening.log", "r") as log_file:

        logs = log_file.read()

        print(logs)

except Exception as e:

    print("Could not read log file:", e)

# ============================================================
# PIPELINE COMPLETED
# ============================================================

print("\n==============================")
print("PIPELINE EXECUTION COMPLETED")
print("==============================")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.1/37.1 MB 38.7 MB/s eta 0:00:00


[12:46:24] SMILES Parse Error: syntax error while parsing: INVALID_SMILES
[12:46:24] SMILES Parse Error: check for mistakes around position 3:
[12:46:24] INVALID_SMILES
[12:46:24] ~~^
[12:46:24] SMILES Parse Error: Failed parsing SMILES 'INVALID_SMILES' for input: 'INVALID_SMILES'
ERROR:root:Validation failed for INVALID_SMILES: Invalid SMILES structure


Logging initialized

STARTING DRUG SCREENING

Processing Molecule: CCO
Result:
{'SMILES': 'CCO', 'Validation': 'Passed', 'Descriptors': {'MolecularWeight': 46.07, 'LogP': -0.0, 'HDonors': 1, 'HAcceptors': 1}, 'LipinskiRule': 'Pass', 'DockingScore': -6.18, 'DockingStatus': 'Success', 'Toxicity': 'Low'}

Processing Molecule: CCN(CC)CC
Result:
{'SMILES': 'CCN(CC)CC', 'Validation': 'Passed', 'Descriptors': {'MolecularWeight': 101.19, 'LogP': 1.35, 'HDonors': 0, 'HAcceptors': 1}, 'LipinskiRule': 'Pass', 'DockingScore': -6.66, 'DockingStatus': 'Success', 'Toxicity': 'High'}

Processing Molecule: INVALID_SMILES
Result:
{'SMILES': 'INVALID_SMILES', 'Validation': 'Failed', 'Error': 'Invalid SMILES structure'}

Processing Molecule: CC(=O)Oc1ccccc1C(=O)O
Result:
{'SMILES': 'CC(=O)Oc1ccccc1C(=O)O', 'Validation': 'Passed', 'Descriptors': {'MolecularWeight': 180.16, 'LogP': 1.31, 'HDonors': 1, 'HAcceptors': 3}, 'LipinskiRule': 'Pass', 'DockingScore': -9.06, 'DockingStatus': 'Success', 'Toxicity': 'M

,SMILES,Validation,Descriptors,LipinskiRule,DockingScore,DockingStatus,Toxicity,Error
0,CCO,Passed,"{'MolecularWeight': 46.07, 'LogP': -0.0, 'HDon...",Pass,-6.18,Success,Low,NaN
1,CCN(CC)CC,Passed,"{'MolecularWeight': 101.19, 'LogP': 1.35, 'HDo...",Pass,-6.66,Success,High,NaN
2,INVALID_SMILES,Failed,NaN,NaN,NaN,NaN,NaN,Invalid SMILES structure
3,CC(=O)Oc1ccccc1C(=O)O,Passed,"{'MolecularWeight': 180.16, 'LogP': 1.31, 'HDo...",Pass,-9.06,Success,Moderate,NaN
4,C1CCCCC1,Passed,"{'MolecularWeight': 84.16, 'LogP': 2.34, 'HDon...",Pass,-11.58,Success,Moderate,NaN
5,CC(C)CC1=CC=C(C=C1)C(C)C(=O)O,Passed,"{'MolecularWeight': 206.28, 'LogP': 3.07, 'HDo...",Pass,-10.38,Success,High,NaN



Results saved as:
drug_screening_results.csv

ERROR LOG CONTENT
Could not read log file: [Errno 2] No such file or directory: 'drug_screening.log'

PIPELINE EXECUTION COMPLETED
